# LLM Factory 2 - New Autogen Pattern

This notebook demonstrates the new LLM Factory 2 implementation using the latest autogen model client pattern with `ChatCompletionClient` interface.

## Features
- ✅ OpenAI provider (fully implemented)
- ⏳ Gemini provider (TODO)
- ⏳ Grok provider (TODO) 
- ⏳ Llama provider (TODO)

## Key Changes from Old Pattern
- Uses `autogen_ext.models.openai.OpenAIChatCompletionClient` instead of `LLMConfig`
- Compatible with `autogen_agentchat.agents.AssistantAgent`
- Supports streaming and tool use

## Setup

Import required libraries and add project root to path

In [2]:
import sys
from pathlib import Path

# Add the project root to Python path
project_root = Path().absolute().parent.parent.parent.parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: /Users/ahmedashraf/Desktop/PhysioAssist/PhysioAssist-AI


In [3]:
from src.app.services.llm_factory import LLMFactory
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console

print("✅ Imports successful!")

✅ Imports successful!


## Check Available Providers

In [4]:
# Check all supported providers
all_providers = LLMFactory.get_supported_providers()
print(f"All supported providers: {all_providers}")

# Check implemented providers
implemented = LLMFactory.get_implemented_providers()
print(f"✅ Implemented providers: {implemented}")

# Check pending providers
pending = [p for p in all_providers if p not in implemented]
print(f"⏳ Pending providers: {pending}")

All supported providers: ['openai', 'gemini', 'grok', 'llama']
✅ Implemented providers: ['openai']
⏳ Pending providers: ['gemini', 'grok', 'llama']


## Example 1: Create OpenAI Model Client

Using the factory to create an OpenAI model client with the new pattern.

In [5]:
# Create OpenAI model client
model_client = LLMFactory.create(
    provider="openai",
    model="gpt-4o-mini",  # Using mini for testing
    temperature=0
)

print(f"✅ Model client created: {type(model_client).__name__}")
print(f"Model: gpt-4o-mini")
print(f"Temperature: 0")

✅ Model client created: OpenAIChatCompletionClient
Model: gpt-4o-mini
Temperature: 0


## Example 2: Simple AssistantAgent

Create a basic assistant agent using the model client.

In [6]:
# Create a simple assistant agent
simple_agent = AssistantAgent(
    name="simple_assistant",
    model_client=model_client,
    system_message="You are a helpful assistant that provides concise answers.",
)

print("✅ Simple assistant agent created!")

✅ Simple assistant agent created!


In [ ]:
# Run a simple task
async def test_simple_agent():
    result = await simple_agent.run(task="What is 2 + 2? Just give me the number.")
    
    # Print the last message content (the assistant's response)
    if result.messages:
        last_message = result.messages[-1]
        print(f"Assistant response: {last_message.content}")
    
    return result

# Execute the task
result = await test_simple_agent()
print(f"\nFull result: {result}")

Assistant response: 4

Full result: messages=[TextMessage(id='67c7b8bb-b777-4547-a014-9f50b1b16cae', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 2, 13, 13, 3, 21, 3937, tzinfo=datetime.timezone.utc), content='What is 2 + 2? Just give me the number.', type='TextMessage'), TextMessage(id='a97a9678-c12f-43c5-95b5-460cf14e9803', source='simple_assistant', models_usage=RequestUsage(prompt_tokens=36, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 2, 13, 13, 3, 22, 661887, tzinfo=datetime.timezone.utc), content='4', type='TextMessage')] stop_reason=None


## Example 3: Agent with Tools

Create an agent with a custom tool (like the weather example from autogen docs).

In [8]:
# Define a simple tool function
async def get_weather(city: str) -> str:
    """Get the weather for a given city."""
    # Mock weather data for demonstration
    weather_data = {
        "New York": "73°F and Sunny",
        "London": "55°F and Rainy", 
        "Tokyo": "68°F and Cloudy",
        "Paris": "62°F and Partly Cloudy",
    }
    return f"The weather in {city} is {weather_data.get(city, '72°F and Clear')}."

# Create a new model client for the weather agent
weather_model_client = LLMFactory.create(
    provider="openai",
    model="gpt-4o-mini",
    temperature=0.2
)

# Create agent with tool
weather_agent = AssistantAgent(
    name="weather_agent",
    model_client=weather_model_client,
    tools=[get_weather],
    system_message="You are a helpful weather assistant. Use the get_weather tool to answer questions about weather.",
    reflect_on_tool_use=True,
)

print("✅ Weather agent with tool created!")

✅ Weather agent with tool created!


In [10]:
# Run the weather agent
async def test_weather_agent():
    result = await weather_agent.run(task="What is the weather in New York?")
    
    # Print the conversation trace
    print("=== Conversation Trace ===")
    for i, message in enumerate(result.messages, 1):
        print(f"\n[Message {i}]")
        print(f"Source: {message.source}")
        print(f"Content: {message.content}")
        if hasattr(message, 'models_usage') and message.models_usage:
            print(f"Tokens: {message.models_usage}")
    
    return result

# Execute the task
result = await test_weather_agent()
print(f"\n=== Stop Reason: {result.stop_reason} ===")

=== Conversation Trace ===

[Message 1]
Source: user
Content: What is the weather in New York?

[Message 2]
Source: weather_agent
Content: [FunctionCall(id='call_AlVSFXgAWzXBR3aJlvAM8ZKh', arguments='{"city":"New York"}', name='get_weather')]
Tokens: RequestUsage(prompt_tokens=143, completion_tokens=15)

[Message 3]
Source: weather_agent
Content: [FunctionExecutionResult(content='The weather in New York is 73°F and Sunny.', name='get_weather', call_id='call_AlVSFXgAWzXBR3aJlvAM8ZKh', is_error=False)]

[Message 4]
Source: weather_agent
Content: The weather in New York is still 73°F and sunny.
Tokens: RequestUsage(prompt_tokens=138, completion_tokens=13)

=== Stop Reason: None ===


### Alternative: Using Console for Streaming

For a nicer real-time display of the conversation, you can use `Console` with `run_stream()`:

In [11]:
# Using Console for streaming output (shows conversation in real-time)
await Console(weather_agent.run_stream(task="What's the weather like in Paris?"))

---------- TextMessage (user) ----------
What's the weather like in Paris?
---------- ToolCallRequestEvent (weather_agent) ----------
[FunctionCall(id='call_Zj8ooLYtULK071iv8bgrOQbo', arguments='{"city":"Paris"}', name='get_weather')]
---------- ToolCallExecutionEvent (weather_agent) ----------
[FunctionExecutionResult(content='The weather in Paris is 62°F and Partly Cloudy.', name='get_weather', call_id='call_Zj8ooLYtULK071iv8bgrOQbo', is_error=False)]
---------- TextMessage (weather_agent) ----------
The weather in Paris is currently 62°F and partly cloudy.


TaskResult(messages=[TextMessage(id='518c29a2-6d19-4637-ac3d-988ea0fda68d', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 2, 13, 13, 6, 2, 779567, tzinfo=datetime.timezone.utc), content="What's the weather like in Paris?", type='TextMessage'), ToolCallRequestEvent(id='94af60fa-4321-4a23-96fe-87bf8039ffe6', source='weather_agent', models_usage=RequestUsage(prompt_tokens=207, completion_tokens=14), metadata={}, created_at=datetime.datetime(2026, 2, 13, 13, 6, 3, 515972, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_Zj8ooLYtULK071iv8bgrOQbo', arguments='{"city":"Paris"}', name='get_weather')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='9299aeee-0506-411f-addb-67f975606a36', source='weather_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 2, 13, 13, 6, 3, 518563, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='The weather in Paris is 62°F and Partly Cloudy.', name='get_weath